In [1]:
# Dependencies (unsloth, unsloth_zoo, sentencepiece, protobuf, hf_transfer,
# xformers, trl, torchvision, transformers, bitsandbytes, accelerate, peft,
# datasets) are managed by UV via `pyproject.toml`.
#
# From the repository root, install them with:
#
#     uv sync --extra unsloth
#
# Then launch Jupyter from the same environment (e.g. `uv run jupyter lab`)
# so this notebook can import unsloth without any in-notebook `pip install`.

In [2]:
from unsloth import FastModel
from transformers import AutoModelForSequenceClassification
import torch

# Compat shim: torchvision cu128 wheels are built without the video backend,
# so `torchvision.io.VideoReader` is missing. `datasets==4.3.0` (pinned by
# unsloth) imports it unconditionally in its Torch formatter, which breaks
# `trainer.train()`. Stub a dummy class so the isinstance(...) branch is a
# harmless no-op (we do not use video in this notebook).
import torchvision.io
if not hasattr(torchvision.io, "VideoReader"):
    class VideoReader:  # noqa: N801 - mirror torchvision's public name
        pass
    torchvision.io.VideoReader = VideoReader

%env UNSLOTH_DISABLE_FAST_GENERATION = 1
max_seq_length = 256
dtype = None
load_in_4bit = False

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
env: UNSLOTH_DISABLE_FAST_GENERATION=1


In [3]:
import os
from dotenv import load_dotenv

load_dotenv()
hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HF_NEW")
assert hf_token, "Set HF_TOKEN (or HF_NEW) in your .env file at the repo root"

# ---- Weights & Biases: comprehensive experiment tracking -------------------
# Follows the official HF + W&B integration guide
# (https://docs.wandb.ai/models/integrations/huggingface_transformers).
import wandb
from datetime import datetime

wandb_api_key = os.environ.get("WANDB_API_KEY")
if wandb_api_key:
    wandb.login(key=wandb_api_key)

# Trainer-level integration knobs (read by transformers.integrations.WandbCallback).
# Must be set BEFORE the Trainer is instantiated.
os.environ.setdefault("WANDB_WATCH", "all")      # gradients + params histograms
os.environ.setdefault("WANDB_LOG_MODEL", "false") # we upload the final model manually
os.environ.setdefault("WANDB_SILENT", "false")

run_name = os.environ.get(
    "WANDB_NAME",
    f"embedding-classifier-e5-large-{datetime.utcnow():%Y%m%d-%H%M%S}",
)

wandb_run = wandb.init(
    project=os.environ.get("WANDB_PROJECT", "lid-bench"),
    entity=os.environ.get("WANDB_ENTITY"),
    name=run_name,
    group="embedding-classifier",
    job_type="train",
    tags=["embedding", "xlm-roberta-large", "multilingual-e5-large", "lid-67"],
    config={
        "model_name": "intfloat/multilingual-e5-large",
        "max_seq_length": max_seq_length,
        "load_in_4bit": load_in_4bit,
        "dtype": str(dtype) if dtype is not None else "auto",
        "dataset": "1024m/LID",
        "dataset_file": "Data_Hackathon/LID-1000.parquet",
    },
    save_code=True,
    reinit=True,
)

# Define how each metric should be summarized in the run summary panel.
# Trainer pushes eval metrics under eval_<dataset>/<metric_name>.
for prefix in ("eval_val",):
    wandb.define_metric(f"{prefix}/macro_f1", summary="max")
    wandb.define_metric(f"{prefix}/micro_f1", summary="max")
    wandb.define_metric(f"{prefix}/weighted_f1", summary="max")
    wandb.define_metric(f"{prefix}/accuracy", summary="max")
    wandb.define_metric(f"{prefix}/macro_precision", summary="max")
    wandb.define_metric(f"{prefix}/macro_recall", summary="max")
    wandb.define_metric(f"{prefix}/loss", summary="min")
wandb.define_metric("train/loss", summary="min")
wandb.define_metric("train/learning_rate", summary="last")

print(f"W&B run: {wandb_run.url}")


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /home/ubuntu/.netrc
wandb: Currently logged in as: cataluna84 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
/tmp/ipykernel_27026/375357118.py:26: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  f"embedding-classifier-e5-large-{datetime.utcnow():%Y%m%d-%H%M%S}",


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


wandb: Detected [huggingface_hub.inference] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


W&B run: https://wandb.ai/cataluna84/VisionInterpretability/runs/j2iobtl8


In [4]:
from datasets import load_dataset
LOAD_SPECIFIC_FILE = True
dataset_name = "1024m/LID"
if LOAD_SPECIFIC_FILE:
    file_path = "Data_Hackathon/LID-1000.parquet"
    dataset = load_dataset("parquet", data_files={"train": f"hf://datasets/{dataset_name}/{file_path}"}, token=hf_token)["train"]
else:
    dataset = load_dataset(dataset_name, token=hf_token)["train"]
print(f"dataset  : {dataset_name}")
print(f"samples  : {len(dataset)}")
print(f"columns  : {dataset.column_names}")
print(f"size     : {dataset.dataset_size / 1024**2:.3f} MB")

dataset  : 1024m/LID
samples  : 67000
columns  : ['lang', 'text', 'source', 'ISO-693-3']
size     : 676.859 MB


In [5]:
NUM_LABELS = len(dataset.unique("ISO-693-3"))
print(NUM_LABELS)
print(dataset.unique("ISO-693-3"))

67
['gle', 'vie', 'por', 'hrv', 'cym', 'ara', 'xho', 'zho', 'nep', 'ces', 'nld', 'urd', 'tel', 'guj', 'lit', 'kor', 'tur', 'pol', 'mlt', 'fra', 'pan', 'msa', 'deu', 'ibo', 'mar', 'hau', 'swe', 'ron', 'rus', 'zul', 'tam', 'yor', 'ben', 'heb', 'est', 'dan', 'srp', 'cat', 'mya', 'lao', 'jpn', 'slv', 'nor', 'bul', 'slk', 'mlg', 'ind', 'fin', 'ell', 'glg', 'swh', 'hin', 'khm', 'jav', 'eus', 'tha', 'eng', 'fas', 'ukr', 'amh', 'wol', 'lav', 'ita', 'spa', 'hun', 'tgl', 'sna']


In [6]:
labels = sorted(dataset.unique("ISO-693-3"))
id2label = {i: l for i, l in enumerate(labels)}
label2id = {l: i for i, l in enumerate(labels)}

In [7]:
import os
os.environ["UNSLOTH_WARN_UNINITIALIZED"] = "0"
import torch.nn as nn
model, tokenizer = FastModel.from_pretrained(
    model_name = "intfloat/multilingual-e5-large",
    auto_model = AutoModelForSequenceClassification,
    max_seq_length = max_seq_length,
    dtype = dtype,
    full_finetuning = True,
    load_in_4bit = load_in_4bit,
)
if hasattr(model.classifier, "out_proj"):
    model.classifier.out_proj = nn.Linear(model.classifier.out_proj.in_features, NUM_LABELS, bias=True).to(model.device).to(model.dtype)
else:
    model.classifier = nn.Linear(model.classifier.in_features, NUM_LABELS, bias=True).to(model.device).to(model.dtype)
model.config.num_labels = NUM_LABELS
model.config.id2label = id2label
model.config.label2id = label2id

# Unsloth patches XLM-Roberta to use the `flex_attention` backend, which
# raises `ValueError: flex_attention does not support dropout` during
# training. `intfloat/multilingual-e5-large` ships with
# `attention_probs_dropout_prob=0.1`, so zero it out on the config *and*
# on every already-instantiated self-attention layer before trainer.train().
model.config.attention_probs_dropout_prob = 0.0
for _m in model.modules():
    if _m.__class__.__name__.endswith("SelfAttention") and hasattr(_m, "dropout") and hasattr(_m.dropout, "p"):
        _m.dropout.p = 0.0

==((====))==  Unsloth 2026.3.11: Fast Xlm_Roberta patching. Transformers: 5.3.0.
   \\   /|    NVIDIA A10. Num GPUs = 1. Max memory: 22.058 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using bfloat16 full finetuning which cuts memory usage by 50%.
To enable float32 training, use `float32_mixed_precision = True` during FastLanguageModel.from_pretrained


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: intfloat/multilingual-e5-large
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.bias          | UNEXPECTED | 
embeddings.position_ids    | UNEXPECTED | 
pooler.dense.weight        | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
wandb: WARNING Artifact "source-VisionInterpretability-_home_ubuntu_Workspace_lid_notebooks_LID_Embedding_Classifier.ipynb" already exists with the same content. No new version will be created.


In [8]:
model = FastModel.get_peft_model(model, r = 8, target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj",],
                                 lora_alpha = 16, lora_dropout = 0, bias = "none", use_gradient_checkpointing = "unsloth",
                                 random_state = 1024, use_rslora = False, loftq_config = None, task_type = "SEQ_CLS",)

Unsloth: Full finetuning is enabled, so .get_peft_model has no effect


In [9]:
from datasets import ClassLabel
if isinstance(dataset, dict):
    dataset = dataset["train"]
dataset = dataset.cast_column("ISO-693-3", ClassLabel(names=sorted(dataset.unique("ISO-693-3"))))
dataset = dataset.train_test_split(test_size=0.1, stratify_by_column="ISO-693-3")
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=max_seq_length)
train_dataset = dataset['train'].map(tokenize_function, batched=True, num_proc=16)
val_dataset = dataset["test"].map(tokenize_function, batched=True, num_proc=16)
print(len(train_dataset))
print(len(val_dataset))

Map (num_proc=16):   0%|          | 0/60300 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/6700 [00:00<?, ? examples/s]

60300
6700


In [10]:
train_dataset = train_dataset.rename_column("ISO-693-3", "label")
val_dataset = val_dataset.rename_column("ISO-693-3", "label")

In [11]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
labels = train_dataset["label"]
class_weights = compute_class_weight("balanced", classes = np.unique(labels), y = labels)

In [12]:
train_dataset = train_dataset.rename_column("label", "labels")
val_dataset = val_dataset.rename_column("label", "labels")
train_dataset = train_dataset.remove_columns(["text", "lang", "source"])
val_dataset = val_dataset.remove_columns(["text", "lang", "source"])
train_dataset.set_format("torch")
val_dataset.set_format("torch")

In [13]:
import json
def process_benchmark(file_path):
    ds = load_dataset("parquet", data_files={"train": f"hf://datasets/{dataset_name}/{file_path}"}, token=hf_token)["train"]
    lang_col = next(c for c in ds.column_names if c.lower() == "iso-693-3")
    text_col = next(c for c in ds.column_names if c.lower() == "text")
    ds = ds.filter(lambda x: x[lang_col] in label2id)
    ds = ds.map(lambda x: {"labels": label2id[x[lang_col]]})
    if text_col != "text":
        ds = ds.rename_column(text_col, "text")
    ds = ds.map(tokenize_function, batched=True, num_proc=16)
    keep = [c for c in ["input_ids", "attention_mask", "token_type_ids", "labels"] if c in ds.column_names]
    ds = ds.remove_columns([c for c in ds.column_names if c not in keep])
    ds.set_format("torch")
    return ds
benchmark_files = {
    "CommonLID": "Data_Benchmarks_Filtered/filtered_benchmark_CommonLID.parquet",
    "FLORES":    "Data_Benchmarks_Filtered/filtered_benchmark_FLORES.parquet",
    "SmolSent":  "Data_Benchmarks_Filtered/filtered_benchmark_SmolSent.parquet",
    "UDHRLID":   "Data_Benchmarks_Filtered/filtered_benchmark_UDHRLID.parquet",
}
benchmark_datasets = {name: process_benchmark(path) for name, path in benchmark_files.items()}
for name, ds in benchmark_datasets.items():
    print(f"{name}: {len(ds)} samples")

CommonLID: 268682 samples
FLORES: 58696 samples
SmolSent: 8630 samples
UDHRLID: 3987 samples


In [14]:
from sklearn.metrics import (
    f1_score, accuracy_score, precision_score, recall_score,
)
import numpy as np
import json
import wandb

eval_context = {"dataset_name": "val", "step": 0}

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)

    metrics = {
        "macro_f1":        f1_score(labels, preds, average="macro", zero_division=0),
        "micro_f1":        f1_score(labels, preds, average="micro", zero_division=0),
        "weighted_f1":     f1_score(labels, preds, average="weighted", zero_division=0),
        "accuracy":        accuracy_score(labels, preds),
        "macro_precision": precision_score(labels, preds, average="macro", zero_division=0),
        "macro_recall":    recall_score(labels, preds, average="macro", zero_division=0),
    }

    # Per-language accuracy (dict -> JSON file + wandb.Table).
    per_label = {}
    per_lang_rows = []
    for lid in np.unique(labels):
        mask = labels == lid
        lang = id2label[int(lid)]
        lang_acc = float(accuracy_score(labels[mask], preds[mask]))
        per_label[lang] = f"{lang_acc:.3f}"
        per_lang_rows.append([lang, lang_acc, int(mask.sum())])

    name = eval_context["dataset_name"]
    step = eval_context["step"]
    with open(f"{name}-{step}-SCORES.json", "w") as f:
        json.dump(per_label, f, indent=2)

    if wandb.run is not None:
        per_lang_table = wandb.Table(
            columns=["language", "accuracy", "n_samples"],
            data=per_lang_rows,
        )
        payload = {
            f"charts/{name}/per_lang_accuracy_table": per_lang_table,
            f"charts/{name}/per_lang_accuracy_bar": wandb.plot.bar(
                per_lang_table, "language", "accuracy",
                title=f"Per-language accuracy ({name}, step {step})",
            ),
        }
        # Confusion matrix is informative but expensive with 67 classes; only
        # log on the primary val set to keep logs lean.
        if name == "val":
            class_names = [id2label[i] for i in range(NUM_LABELS)]
            payload[f"charts/{name}/confusion_matrix"] = wandb.plot.confusion_matrix(
                y_true=labels.tolist(),
                preds=preds.tolist(),
                class_names=class_names,
                title=f"Confusion matrix ({name}, step {step})",
            )
        wandb.log(payload, step=step)

    return metrics


In [15]:
from transformers import TrainingArguments, Trainer
from unsloth import is_bfloat16_supported
import torch

class LIDTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.functional.cross_entropy(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

    def evaluate(self, eval_dataset=None, ignore_keys=None, metric_key_prefix="eval"):
        eval_context["step"] = self.state.global_step
        if eval_dataset is None and isinstance(self.eval_dataset, dict):
            all_metrics = {}
            for name, ds in self.eval_dataset.items():
                eval_context["dataset_name"] = name
                m = super().evaluate(
                    eval_dataset=ds,
                    ignore_keys=ignore_keys,
                    metric_key_prefix=f"eval_{name}",
                )
                all_metrics.update(m)
            return all_metrics
        eval_context["dataset_name"] = metric_key_prefix
        return super().evaluate(
            eval_dataset=eval_dataset,
            ignore_keys=ignore_keys,
            metric_key_prefix=metric_key_prefix,
        )

eval_datasets = {"val": val_dataset, **benchmark_datasets}

trainer = LIDTrainer(
    model=model,
    processing_class=tokenizer,
    eval_dataset=eval_datasets,
    train_dataset=train_dataset,
    args=TrainingArguments(
        # Memory-tuned for ~24 GB GPU (original A100-80GB config was
        # per_device_train_batch_size=1440 / grad_accum=1). We preserve the
        # effective batch (32 * 45 = 1440) via gradient accumulation and
        # enable gradient checkpointing to shrink activation memory ~10x.
        per_device_train_batch_size=32,
        per_device_eval_batch_size=32,
        gradient_accumulation_steps=45,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.001,
        eval_strategy="steps",
        eval_steps=0.1,
        lr_scheduler_type="linear",
        seed=1024,
        output_dir="outputs",
        # Full W&B logging (docs: "the most important step"):
        report_to="wandb",
        run_name=wandb_run.name if wandb_run is not None else None,
        logging_dir="outputs/runs",
        log_level="info",
    ),
    compute_metrics=compute_metrics,
)

# Runtime-derived config not available at wandb.init() time.
if wandb.run is not None:
    wandb.config.update({
        "effective_batch_size": trainer.args.per_device_train_batch_size
                                 * trainer.args.gradient_accumulation_steps,
        "num_labels": NUM_LABELS,
        "train_samples": len(train_dataset),
        "val_samples": len(val_dataset),
        **{f"benchmark_samples/{k}": len(v) for k, v in benchmark_datasets.items()},
        "class_weights_mean": float(np.mean(class_weights)),
        "class_weights_min": float(np.min(class_weights)),
        "class_weights_max": float(np.max(class_weights)),
    }, allow_val_change=True)


`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [16]:
trainer_stats = trainer.train()

# Final summary: pin the most important numbers to the run summary panel
# so they show up in the W&B Runs table.
if wandb.run is not None:
    summary = {
        "final/train_runtime_sec":         float(trainer_stats.metrics.get("train_runtime", 0.0)),
        "final/train_samples_per_second":  float(trainer_stats.metrics.get("train_samples_per_second", 0.0)),
        "final/train_steps_per_second":    float(trainer_stats.metrics.get("train_steps_per_second", 0.0)),
        "final/train_loss":                float(trainer_stats.training_loss),
        "final/total_flos":                float(trainer_stats.metrics.get("total_flos", 0.0)),
        "final/epoch":                     float(trainer_stats.metrics.get("epoch", 0.0)),
    }
    if torch.cuda.is_available():
        summary["final/gpu_mem_peak_mb"]     = torch.cuda.max_memory_allocated() / 1e6
        summary["final/gpu_mem_reserved_mb"] = torch.cuda.max_memory_reserved() / 1e6
    wandb.run.summary.update(summary)


skipped Embedding(250002, 1024, padding_idx=1): 244.142578125M params
skipped Embedding(1, 1024): 244.1435546875M params
skipped Embedding(514, 1024, padding_idx=1): 244.6455078125M params
skipped: 244.6455078125M params
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 60,300 | Num Epochs = 1 | Total steps = 42
O^O/ \_/ \    Batch size per device = 32 | Gradient accumulation steps = 45
\        /    Data Parallel GPUs = 1 | Total batch size (32 x 45 x 1) = 1,440
 "-____-"     Trainable parameters = 559,959,107 of 559,959,107 (100.00% trained)


Step,Training Loss,Validation Loss,Val Loss,Val Macro F1,Val Micro F1,Val Weighted F1,Val Accuracy,Val Macro Precision,Val Macro Recall,Commonlid Loss,Commonlid Macro F1,Commonlid Micro F1,Commonlid Weighted F1,Commonlid Accuracy,Commonlid Macro Precision,Commonlid Macro Recall,Flores Loss,Flores Macro F1,Flores Micro F1,Flores Weighted F1,Flores Accuracy,Flores Macro Precision,Flores Macro Recall,Smolsent Loss,Smolsent Macro F1,Smolsent Micro F1,Smolsent Weighted F1,Smolsent Accuracy,Smolsent Macro Precision,Smolsent Macro Recall,Udhrlid Loss,Udhrlid Macro F1,Udhrlid Micro F1,Udhrlid Weighted F1,Udhrlid Accuracy,Udhrlid Macro Precision,Udhrlid Macro Recall
5,2.764128,No log,2.202143,0.954219,0.957164,0.954219,0.957164,0.966654,0.957164,2.755620,0.559772,0.735360,0.774484,0.735360,0.594996,0.615858,2.546642,0.800808,0.907319,0.900891,0.907319,0.823354,0.806633,2.799616,0.503434,0.831750,0.805494,0.831750,0.529202,0.519844,2.543567,0.846196,0.909456,0.906045,0.909456,0.861560,0.851191
10,0.780083,No log,0.405896,0.984233,0.984478,0.984233,0.984478,0.984706,0.984478,0.988169,0.605123,0.914341,0.937687,0.914341,0.599711,0.670237,0.628206,0.963224,0.980390,0.980465,0.980390,0.965167,0.963149,1.011914,0.705375,0.915759,0.916987,0.915759,0.706457,0.704430,0.699225,0.938645,0.950840,0.951004,0.950840,0.947947,0.944354
15,0.222642,No log,0.125282,0.986797,0.987015,0.986797,0.987015,0.987378,0.987015,0.459381,0.614483,0.928771,0.943809,0.928771,0.602116,0.687003,0.180303,0.951723,0.985553,0.985373,0.985553,0.953381,0.951899,0.364460,0.779575,0.935110,0.935490,0.935110,0.780070,0.779258,0.279313,0.941667,0.952847,0.953841,0.952847,0.952261,0.946648
20,0.106136,No log,0.069874,0.986657,0.987015,0.986657,0.987015,0.987436,0.987015,0.371983,0.617309,0.921547,0.939143,0.921547,0.612701,0.690129,0.099693,0.944473,0.981958,0.977998,0.981958,0.957456,0.948365,0.241239,0.722132,0.898957,0.866558,0.898957,0.791208,0.749131,0.219471,0.936198,0.950339,0.949067,0.950339,0.956150,0.943765
25,0.088826,No log,0.054392,0.990057,0.990149,0.990057,0.990149,0.990159,0.990149,0.347588,0.623296,0.922246,0.938305,0.922246,0.614834,0.694448,0.081897,0.955822,0.988824,0.989543,0.988824,0.959907,0.955115,0.192509,0.771477,0.933024,0.925772,0.933024,0.799018,0.777520,0.210583,0.950026,0.958866,0.960922,0.958866,0.960497,0.953720
30,0.067940,No log,0.048404,0.990497,0.990597,0.990497,0.990597,0.990603,0.990597,0.337770,0.623685,0.922808,0.939690,0.922808,0.615009,0.695057,0.070972,0.957146,0.990476,0.990890,0.990476,0.960164,0.956739,0.176348,0.850346,0.940324,0.935381,0.940324,0.874062,0.854840,0.201208,0.950280,0.959117,0.961146,0.959117,0.960035,0.954003
35,0.069293,No log,0.046209,0.991062,0.991194,0.991062,0.991194,0.991365,0.991194,0.342643,0.623432,0.921268,0.939050,0.921268,0.614919,0.694779,0.068014,0.957059,0.990545,0.990802,0.990545,0.960081,0.956806,0.173134,0.849751,0.939861,0.934726,0.939861,0.873881,0.854419,0.197606,0.950390,0.959368,0.961253,0.959368,0.959828,0.954275
40,0.065116,No log,0.045676,0.991379,0.991493,0.991379,0.991493,0.991618,0.991493,0.344161,0.622955,0.920787,0.938794,0.920787,0.614216,0.694833,0.067222,0.957459,0.990936,0.991209,0.990936,0.960242,0.957192,0.170790,0.781850,0.942526,0.938220,0.942526,0.802048,0.785438,0.196806,0.950390,0.959368,0.961253,0.959368,0.959828,0.954275



***** Running Evaluation *****
  Num examples = 6700
  Batch size = 32

***** Running Evaluation *****
  Num examples = 268682
  Batch size = 32

***** Running Evaluation *****
  Num examples = 58696
  Batch size = 32
wandb: WARNING Tried to log to step 5 that is less than the current step 6. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.

***** Running Evaluation *****
  Num examples = 8630
  Batch size = 32
wandb: WARNING Tried to log to step 5 that is less than the current step 7. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.

***** Running Evaluation *****
  Num examples = 3987
  Batch size = 32
wandb: WARNING Tried to log to step 5 that is less than the current step 8. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.
wandb: WARNING T

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model weights saved in outputs/checkpoint-42/model.safetensors


Training completed. Do not forget to share your model on huggingface.co/models =)




In [17]:
from transformers import pipeline

classifier = pipeline("text-classification", model=model, tokenizer=tokenizer)
test_text = dataset["test"][0]["text"]
true_label = id2label[int(val_dataset[0]["labels"])]
result = classifier(test_text, truncation=True, max_length=256)
print(f"text       : {test_text[:100]}")
print(f"true label : {true_label}")
print(f"predicted  : {result[0]['label']} ({result[0]['score']:.4f})")

if wandb.run is not None:
    sample_table = wandb.Table(
        columns=["text", "true_label", "predicted_label", "confidence"],
        data=[[test_text[:500], true_label, result[0]["label"], float(result[0]["score"])]],
    )
    wandb.log({"inference/sanity_check": sample_table})


text       : C5 kan syfta på

 C6 – en kuvertstorlek, se C-format
 C5 (tunnelbanevagn) – en typ av tunnelbanevagn
true label : swe
predicted  : swe (0.9463)


In [18]:
save_dir = "baseline_lora_v2"
model.save_pretrained(save_dir)  # Local saving
tokenizer.save_pretrained(save_dir)

if wandb.run is not None:
    artifact = wandb.Artifact(
        name=f"model-{wandb.run.name}",
        type="model",
        description="Fine-tuned multilingual-e5-large for 67-language LID",
        metadata={
            "base_model":      "intfloat/multilingual-e5-large",
            "num_labels":      NUM_LABELS,
            "max_seq_length":  max_seq_length,
            "effective_batch": 32 * 45,
            "final_train_loss": float(trainer_stats.training_loss),
        },
    )
    artifact.add_dir(save_dir)
    wandb.run.log_artifact(artifact, aliases=["latest", "final"])


Configuration saved in baseline_lora_v2/config.json


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model weights saved in baseline_lora_v2/model.safetensors
wandb: Adding directory to artifact (baseline_lora_v2)... Done. 2.6s


In [20]:
model.push_to_hub("cataluna84/LIDL1Bv3", token=hf_token)       # Online saving
tokenizer.push_to_hub("cataluna84/LIDL1Bv3", token=hf_token)   # Online saving

if wandb.run is not None:
    wandb.run.summary["hf_model_url"] = "https://huggingface.co/cataluna84/LIDL1Bv3"
    wandb.finish()   # Required in notebooks to close the run cleanly.


README.md:   0%|          | 0.00/564 [00:00<?, ?B/s]

Configuration saved in /tmp/tmptm1abn33/config.json


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model weights saved in /tmp/tmptm1abn33/model.safetensors
Uploading the following files to cataluna84/LIDL1Bv3: README.md,config.json,model.safetensors


Processing Files (0 / 0): |          |  0.00B /  0.00B            

...

Saved model to https://huggingface.co/cataluna84/LIDL1Bv3


Uploading the following files to cataluna84/LIDL1Bv3: tokenizer.json,README.md,tokenizer_config.json


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

eval/CommonLID_accuracy,▁▇██████
eval/CommonLID_loss,█▃▁▁▁▁▁▁
eval/CommonLID_macro_f1,▁▆▇▇████
eval/CommonLID_macro_precision,▁▃▃▇████
eval/CommonLID_macro_recall,▁▆▇█████
eval/CommonLID_micro_f1,▁▇██████
eval/CommonLID_runtime,▆▂▄▂▇█▃▁
eval/CommonLID_samples_per_second,▃▇▄▇▂▁▆█
eval/CommonLID_steps_per_second,▃▇▄▇▂▁▆█
eval/CommonLID_weighted_f1,▁███████
+45,...
